# *General Information*
## Game Mechanics

- **Planets** produce ships each turn (proportional to their radius)
- **Inner planets** rotate around the central sun; outer planets are static
- **Fleets** fly in straight lines at a given angle from their source planet
- **Fleet speed** scales with fleet size (1 ship = 1/turn, larger fleets up to 6/turn)
- **Combat**: arriving fleet ships are subtracted from the planet's garrison. If the garrison drops below 0, ownership flips
- **Sun**: fleets that hit the sun are destroyed
- **Comets**: temporary planets that fly through the board on elliptical paths
- **Win condition**: most ships (planets + fleets) when time runs out or only one player remains

## Action Dictionary
0. Do nothing
1. Attack nearest neutral 
2. Attack weakest neutral 
3. Attack weakest enemy
4. Attack highest production target
5. Reinforce weakest friendly

# **Installation and Versions**

In [76]:
!pip install -q gymnasium stable-baselines3
!pip install "kaggle-environments>=1.28.0"

In [77]:
### import kaggle_environments
from kaggle_environments import make

import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

def parse_version(text):
    parts = []
    for token in text.split('.'):
        digits = ''.join(ch for ch in token if ch.isdigit())
        parts.append(int(digits or 0))
    return tuple(parts)

required_version = (1, 28, 0)
needs_upgrade = False

try:
    installed_version = parse_version(version('kaggle-environments'))
    needs_upgrade = installed_version < required_version
except PackageNotFoundError:
    needs_upgrade = True

if needs_upgrade:
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'kaggle-environments>=1.28.0']
    )

import kaggle_environments
from kaggle_environments import make
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet

# **Helper Functions**

In [78]:
# Helpers + robust policy primitives
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet

# ---------------------------------------------------------------------
# Game / model constants
# ---------------------------------------------------------------------
SUN_X = 50.0
SUN_Y = 50.0
SUN_RADIUS = 10.0
SUN_BUFFER = 5.0          # conservative no-fly buffer around sun
SUN_SAFETY_MARGIN = SUN_BUFFER
INNER_ORBIT_RADIUS = 30.0

MIN_SEND = 20
SOURCE_SEND_FRACTION = 0.50

# Robust trajectory scan settings
PLANET_HIT_RADIUS = 4.0
ANGLE_SEARCH_DEGREES = 90
ANGLE_STEP_DEGREES = 1.0
DT = 0.25
MAX_SIM_TURNS = 80
MAX_REASONABLE_TRAVEL_TIME = 45

# Policy constants
BASE_RESERVE = 20
CORE_RESERVE_BONUS = 15
NEAR_ENEMY_RESERVE_BONUS = 10
HIGH_PROD_RESERVE_BONUS = 8
MAX_FLEET_FRACTION = 0.65

NUM_GLOBAL_FEATURES = 9
NUM_TARGETS = 3
FEATURES_PER_TARGET = 6
OBS_SIZE = NUM_GLOBAL_FEATURES + NUM_TARGETS * FEATURES_PER_TARGET

ACTION_MEANINGS = {
    0: "do_nothing",
    1: "attack_nearest_neutral",
    2: "attack_weakest_neutral",
    3: "attack_weakest_enemy",
    4: "attack_highest_production_any",
    5: "reinforce_weakest_friendly",
    6: "attack_nearest_enemy",
    7: "attack_highest_production_neutral",
    8: "attack_highest_production_enemy",
    9: "attack_weakest_any",
}

FALLBACK_ORDERS = {
    0: [0],
    1: [1, 2, 7, 4, 5, 0],
    2: [2, 1, 7, 4, 5, 0],
    3: [3, 6, 8, 4, 9, 5, 0],
    4: [4, 7, 8, 9, 5, 0],
    5: [5, 2, 1, 0],
    6: [6, 3, 8, 4, 9, 5, 0],
    7: [7, 2, 1, 4, 5, 0],
    8: [8, 3, 6, 4, 9, 5, 0],
    9: [9, 2, 3, 4, 5, 0],
}

# ---------------------------------------------------------------------
# Observation model
# ---------------------------------------------------------------------
@dataclass
class GameState:
    player: int
    planets: List[Planet]
    mine: List[Planet]
    enemy: List[Planet]
    neutral: List[Planet]
    angular_velocity: float


def to_planet(raw_planet) -> Planet:
    return raw_planet if isinstance(raw_planet, Planet) else Planet(*raw_planet)


def get_state(obs) -> GameState:
    if isinstance(obs, dict):
        player = obs.get("player", 0)
        raw_planets = obs.get("planets", [])
        angular_velocity = obs.get("angular_velocity", 0.0)
    else:
        player = obs.player
        raw_planets = obs.planets
        angular_velocity = getattr(obs, "angular_velocity", 0.0)

    planets = [to_planet(p) for p in raw_planets]
    mine = [p for p in planets if p.owner == player]
    enemy = [p for p in planets if p.owner >= 0 and p.owner != player]
    neutral = [p for p in planets if p.owner == -1]
    return GameState(player, planets, mine, enemy, neutral, angular_velocity)


def parse_obs(obs) -> Tuple[int, List[Planet], List[Planet], List[Planet], List[Planet], float]:
    state = get_state(obs)
    return state.player, state.planets, state.mine, state.enemy, state.neutral, state.angular_velocity

# ---------------------------------------------------------------------
# Geometry / robust trajectory simulation
# ---------------------------------------------------------------------
def distance(a, b) -> float:
    return math.hypot(a.x - b.x, a.y - b.y)


def angle_between(source, x: float, y: float) -> float:
    return math.atan2(y - source.y, x - source.x)


def fleet_speed(num_ships: int) -> float:
    """
    Isolated approximation for Orbit Wars fleet speed.
    If trajectories are early/late, tune only this function.
    """
    return min(6.0, max(1.0, math.sqrt(max(1, int(num_ships)))))


def point_to_segment_distance(px, py, ax, ay, bx, by) -> float:
    abx, aby = bx - ax, by - ay
    apx, apy = px - ax, py - ay
    ab2 = abx * abx + aby * aby
    if ab2 < 1e-12:
        return math.hypot(px - ax, py - ay)
    t = (apx * abx + apy * aby) / ab2
    t = max(0.0, min(1.0, t))
    return math.hypot(px - (ax + t * abx), py - (ay + t * aby))


def point_hits_sun(x, y) -> bool:
    return math.hypot(x - SUN_X, y - SUN_Y) <= SUN_RADIUS + SUN_BUFFER


def segment_hits_sun(x1, y1, x2, y2) -> bool:
    return point_to_segment_distance(SUN_X, SUN_Y, x1, y1, x2, y2) <= SUN_RADIUS + SUN_BUFFER


def shot_hits_sun(source_x, source_y, target_x, target_y) -> bool:
    return segment_hits_sun(source_x, source_y, target_x, target_y)


def direct_shot_hits_sun(source_x, source_y, target_x, target_y) -> bool:
    return shot_hits_sun(source_x, source_y, target_x, target_y)


def direct_shot_is_safe(source_x, source_y, target_x, target_y) -> bool:
    return not shot_hits_sun(source_x, source_y, target_x, target_y)


def predict_planet_position(planet, turns: float, angular_velocity: float) -> Tuple[float, float]:
    dx = planet.x - SUN_X
    dy = planet.y - SUN_Y
    radius = math.hypot(dx, dy)
    if radius < 1e-8 or radius >= INNER_ORBIT_RADIUS:
        return planet.x, planet.y

    theta0 = math.atan2(dy, dx)
    # If the render shows inner planets rotating the other way, flip this sign.
    theta = theta0 - angular_velocity * turns
    return SUN_X + radius * math.cos(theta), SUN_Y + radius * math.sin(theta)


def simulate_shot(source, target, angle: float, ships_sent: int, angular_velocity: float) -> Dict:
    speed = fleet_speed(ships_sent)
    prev_x, prev_y = source.x, source.y
    min_dist = float("inf")
    best_t = None
    best_data = {"fleet_x": source.x, "fleet_y": source.y, "target_x": target.x, "target_y": target.y}

    cos_a, sin_a = math.cos(angle), math.sin(angle)
    t = 0.0
    while t <= MAX_SIM_TURNS:
        fleet_x = source.x + speed * t * cos_a
        fleet_y = source.y + speed * t * sin_a

        if segment_hits_sun(prev_x, prev_y, fleet_x, fleet_y):
            return {
                "hit": False,
                "sun_collision": True,
                "min_dist": min_dist,
                "best_t": best_t,
                "fleet_x": fleet_x,
                "fleet_y": fleet_y,
                "target_x": None,
                "target_y": None,
            }

        target_x, target_y = predict_planet_position(target, t, angular_velocity)
        d = math.hypot(fleet_x - target_x, fleet_y - target_y)

        if d < min_dist:
            min_dist = d
            best_t = t
            best_data = {
                "fleet_x": fleet_x,
                "fleet_y": fleet_y,
                "target_x": target_x,
                "target_y": target_y,
            }

        if d <= PLANET_HIT_RADIUS:
            return {
                "hit": True,
                "sun_collision": False,
                "min_dist": d,
                "best_t": t,
                **best_data,
            }

        prev_x, prev_y = fleet_x, fleet_y
        t += DT

    return {
        "hit": False,
        "sun_collision": False,
        "min_dist": min_dist,
        "best_t": best_t,
        **best_data,
    }


def find_safe_hitting_angle(source, target, ships_sent: int, angular_velocity: float):
    """
    Robust replacement for closed-form intercept + safe_aim_angle.
    It only returns an angle if simulation says the shot hits and avoids the sun.
    """
    direct_angle = math.atan2(target.y - source.y, target.x - source.x)
    offsets = np.arange(-ANGLE_SEARCH_DEGREES, ANGLE_SEARCH_DEGREES + ANGLE_STEP_DEGREES, ANGLE_STEP_DEGREES)
    offsets = sorted(offsets, key=lambda deg: abs(deg))

    best_hit = None
    best_near_miss = None
    for deg in offsets:
        angle = direct_angle + math.radians(float(deg))
        result = simulate_shot(source, target, angle, ships_sent, angular_velocity)
        if result["sun_collision"]:
            continue

        if best_near_miss is None or result["min_dist"] < best_near_miss["result"]["min_dist"]:
            best_near_miss = {"angle": angle, "result": result}

        if result["hit"]:
            if best_hit is None or result["best_t"] < best_hit["result"]["best_t"]:
                best_hit = {"angle": angle, "result": result}

    if best_hit is not None:
        return best_hit["angle"], best_hit["result"]
    return None, best_near_miss["result"] if best_near_miss else None

# Backward-compatible approximate functions. New agents should NOT rely on safe_aim_angle.
def intercept(source, target, ships_sent: int, angular_velocity: float, iterations: int = 6):
    speed = fleet_speed(ships_sent)
    future_x, future_y = target.x, target.y
    travel_time = math.hypot(future_x - source.x, future_y - source.y) / speed
    for _ in range(iterations):
        future_x, future_y = predict_planet_position(target, travel_time, angular_velocity)
        travel_time = math.hypot(future_x - source.x, future_y - source.y) / speed
    return angle_between(source, future_x, future_y), travel_time, future_x, future_y


def intercept_angle_and_time(source, target, ships_sent, angular_velocity, iterations=6):
    return intercept(source, target, ships_sent, angular_velocity, iterations)


def safe_aim_angle(source, target_x: float, target_y: float, base_angle: float) -> Optional[float]:
    # Kept only for old cells. The policy agents below use find_safe_hitting_angle instead.
    if direct_shot_is_safe(source.x, source.y, target_x, target_y):
        return base_angle
    return None

# ---------------------------------------------------------------------
# Policy scoring and move validation
# ---------------------------------------------------------------------
def strongest_my_planet(my_planets: Sequence[Planet]) -> Optional[Planet]:
    return max(my_planets, key=lambda p: p.ships, default=None)


def required_reserve(source, state: GameState) -> int:
    reserve = BASE_RESERVE

    if state.mine and source.id == max(state.mine, key=lambda p: p.ships).id:
        reserve += CORE_RESERVE_BONUS

    if source.production >= 3:
        reserve += HIGH_PROD_RESERVE_BONUS

    nearby_enemies = [e for e in state.enemy if distance(source, e) <= 28]
    if nearby_enemies:
        reserve += NEAR_ENEMY_RESERVE_BONUS
        reserve += int(0.10 * max(e.ships for e in nearby_enemies))

    return min(int(source.ships), int(reserve))


def usable_ships(source, state: GameState) -> int:
    reserve = required_reserve(source, state)
    cap = int(source.ships * MAX_FLEET_FRACTION)
    return max(0, min(int(source.ships - reserve), cap))


def estimate_ships_needed(target, travel_time: float, extra: int = 2) -> int:
    future_garrison = target.ships
    if target.owner != -1:
        future_garrison += target.production * travel_time
    return max(MIN_SEND, int(math.ceil(future_garrison + extra)))


def estimate_ships_needed_for_target(target, travel_time: float, extra: int = 3) -> int:
    return estimate_ships_needed(target, travel_time, extra=extra)


def enemy_reinforcement_risk(target, state: GameState, travel_time: float) -> float:
    risk = 0.0
    for e in state.enemy:
        if e.id == target.id:
            continue
        d = distance(e, target)
        # simple proxy: close high-ship planets can reinforce quickly
        if d / max(1.0, fleet_speed(max(MIN_SEND, e.ships))) <= travel_time + 4:
            risk += 0.5 * e.ships + 2.0 * e.production
    return risk


def validate_attack_candidate(state: GameState, source, target, policy_type: str, base_score: float = 0.0, extra: int = 3) -> Optional[Dict]:
    """
    All offensive moves pass through this validator.
    It rejects moves unless the simulated path hits, avoids sun, and leaves reserve.
    """
    allowed = usable_ships(source, state)
    if allowed < MIN_SEND:
        return None

    rough_time = max(1.0, distance(source, target) / fleet_speed(MIN_SEND))
    ships = estimate_ships_needed(target, rough_time, extra=extra)
    ships = int(max(MIN_SEND, ships))

    if ships > allowed:
        return None

    angle, result = find_safe_hitting_angle(source, target, ships, state.angular_velocity)
    if angle is None or result is None:
        return None
    if result.get("sun_collision") or not result.get("hit"):
        return None
    if result.get("best_t") is None or result["best_t"] > MAX_REASONABLE_TRAVEL_TIME:
        return None

    risk = enemy_reinforcement_risk(target, state, result["best_t"])
    score = base_score - 1.5 * result["best_t"] - 1.0 * ships - 0.25 * risk - 2.0 * result["min_dist"]

    return {
        "type": policy_type,
        "source": source,
        "target": target,
        "ships": ships,
        "angle": angle,
        "score": score,
        "travel_time": result["best_t"],
        "risk": risk,
        "result": result,
    }


def candidate_sources(my_planets: Sequence[Planet], target, state: Optional[GameState] = None) -> List[Planet]:
    usable = []
    for p in my_planets:
        if state is None:
            ok = p.ships >= MIN_SEND
        else:
            ok = usable_ships(p, state) >= MIN_SEND
        if ok:
            usable.append(p)
    return sorted(usable, key=lambda p: (distance(p, target), -p.ships))


def target_pool(state: GameState, owner_filter: str) -> List[Planet]:
    if owner_filter == "neutral":
        return state.neutral
    if owner_filter == "enemy":
        return state.enemy
    if owner_filter == "any":
        return state.enemy + state.neutral
    return []


def select_target(state: GameState, action: int, anchor: Optional[Planet] = None) -> Optional[Planet]:
    action = int(action)
    if action == 1:
        pool = target_pool(state, "neutral")
        return min(pool, key=lambda p: distance(anchor, p), default=None) if anchor else None
    if action == 2:
        return min(target_pool(state, "neutral"), key=lambda p: (p.ships, -p.production), default=None)
    if action == 3:
        return min(target_pool(state, "enemy"), key=lambda p: (p.ships, -p.production), default=None)
    if action == 4:
        return max(target_pool(state, "any"), key=lambda p: (p.production, -p.ships), default=None)
    if action == 6:
        pool = target_pool(state, "enemy")
        return min(pool, key=lambda p: distance(anchor, p), default=None) if anchor else None
    if action == 7:
        return max(target_pool(state, "neutral"), key=lambda p: (p.production, -p.ships), default=None)
    if action == 8:
        return max(target_pool(state, "enemy"), key=lambda p: (p.production, -p.ships), default=None)
    if action == 9:
        return min(target_pool(state, "any"), key=lambda p: (p.ships, -p.production), default=None)
    return None


def build_attack_move(my_planets: Sequence[Planet], target, angular_velocity: float, state: Optional[GameState] = None) -> List[List[float]]:
    if target is None:
        return []
    if state is None:
        # Compatibility path; create a lightweight fake state with these planets.
        state = GameState(player=-999, planets=list(my_planets) + [target], mine=list(my_planets), enemy=[] if target.owner == -1 else [target], neutral=[target] if target.owner == -1 else [], angular_velocity=angular_velocity)

    best = None
    for source in candidate_sources(my_planets, target, state):
        base = 4.0 * target.production - 1.0 * target.ships
        cand = validate_attack_candidate(state, source, target, "macro_attack", base_score=base, extra=3)
        if cand and (best is None or cand["score"] > best["score"]):
            best = cand

    if best is None:
        return []
    return [[best["source"].id, best["angle"], best["ships"]]]


def build_reinforce_move(my_planets: Sequence[Planet]) -> List[List[float]]:
    if len(my_planets) < 2:
        return []
    target = min(my_planets, key=lambda p: p.ships)
    sources = [p for p in my_planets if p.id != target.id and p.ships >= MIN_SEND + BASE_RESERVE]
    sources = sorted(sources, key=lambda p: (-p.ships, distance(p, target)))
    for source in sources:
        ships_to_send = max(MIN_SEND, int((source.ships - BASE_RESERVE) * 0.50))
        if ships_to_send < MIN_SEND:
            continue
        if not direct_shot_is_safe(source.x, source.y, target.x, target.y):
            continue
        return [[source.id, angle_between(source, target.x, target.y), ships_to_send]]
    return []

# ---------------------------------------------------------------------
# Candidate generators for policy bot
# ---------------------------------------------------------------------
def generate_expansion_candidates(state: GameState) -> List[Dict]:
    candidates = []
    for target in state.neutral:
        for source in candidate_sources(state.mine, target, state)[:3]:
            base = 5.0 * target.production - 1.2 * target.ships - 0.8 * distance(source, target)
            cand = validate_attack_candidate(state, source, target, "expand", base_score=base, extra=3)
            if cand:
                candidates.append(cand)
    return candidates


def generate_frontier_attack_candidates(state: GameState) -> List[Dict]:
    candidates = []
    if not state.enemy:
        return candidates
    enemy_core = max(state.enemy, key=lambda p: p.ships)
    for target in state.enemy:
        # Avoid heroic attacks on the enemy core unless it is already weak.
        if target.id == enemy_core.id and target.ships > 45:
            continue
        for source in candidate_sources(state.mine, target, state)[:3]:
            base = 6.0 * target.production + 2.0 * target.ships - 1.0 * distance(source, target)
            cand = validate_attack_candidate(state, source, target, "frontier_attack", base_score=base, extra=5)
            if cand:
                candidates.append(cand)
    return candidates


def choose_non_conflicting_moves(candidates: List[Dict], max_moves: int = 2) -> List[List[float]]:
    moves = []
    used_sources = set()
    for cand in sorted(candidates, key=lambda c: c["score"], reverse=True):
        sid = cand["source"].id
        if sid in used_sources:
            continue
        used_sources.add(sid)
        moves.append([sid, cand["angle"], cand["ships"]])
        if len(moves) >= max_moves:
            break
    return moves

# ---------------------------------------------------------------------
# Macro-action API used by RL wrapper and simple agents
# ---------------------------------------------------------------------
def choose_move_from_macro_action(obs, action: int) -> List[List[float]]:
    state = get_state(obs)
    if int(action) == 0 or not state.mine:
        return []
    if int(action) == 5:
        return build_reinforce_move(state.mine)
    anchor = strongest_my_planet(state.mine)
    target = select_target(state, int(action), anchor)
    return build_attack_move(state.mine, target, state.angular_velocity, state=state)


def choose_move_with_fallback(obs, chosen_action: int):
    attempted = []
    for action in FALLBACK_ORDERS.get(int(chosen_action), [0]):
        attempted.append(action)
        move = choose_move_from_macro_action(obs, action)
        if move or action == 0:
            used_fallback = action != int(chosen_action)
            failed_original = int(chosen_action) != 0 and action != int(chosen_action)
            return move, action, used_fallback, failed_original, attempted
    return [], 0, True, True, attempted


def macro_agent_factory(action: int):
    def agent(obs, config=None):
        return choose_move_from_macro_action(obs, action)
    return agent


def random_like_macro_agent(obs, config=None):
    action = random.randint(0, len(ACTION_MEANINGS) - 1)
    move, _, _, _, _ = choose_move_with_fallback(obs, action)
    return move

# ---------------------------------------------------------------------
# RL feature extraction
# ---------------------------------------------------------------------
def extract_features(obs) -> np.ndarray:
    state = get_state(obs)
    my_total_ships = sum(p.ships for p in state.mine)
    enemy_total_ships = sum(p.ships for p in state.enemy)
    neutral_total_ships = sum(p.ships for p in state.neutral)
    my_total_prod = sum(p.production for p in state.mine)
    enemy_total_prod = sum(p.production for p in state.enemy)
    neutral_total_prod = sum(p.production for p in state.neutral)

    features = [
        float(len(state.mine)), float(len(state.enemy)), float(len(state.neutral)),
        float(my_total_ships), float(enemy_total_ships), float(neutral_total_ships),
        float(my_total_prod), float(enemy_total_prod), float(neutral_total_prod),
    ]

    anchor = strongest_my_planet(state.mine)
    candidates = sorted(state.enemy + state.neutral, key=lambda p: (p.owner == -1, p.ships, -p.production))[:NUM_TARGETS]
    for planet in candidates:
        features.extend([
            float(planet.owner), float(planet.ships), float(planet.production),
            float(planet.x), float(planet.y), float(distance(anchor, planet) if anchor else 0.0),
        ])
    while len(features) < OBS_SIZE:
        features.extend([0.0] * FEATURES_PER_TARGET)
    return np.array(features[:OBS_SIZE], dtype=np.float32)


# **Simpleton**

In [79]:
#Simpleton

import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet

def Simpleton(obs):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]

    # Separate our planets from targets
    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]

    if not targets:
        return moves

    for mine in my_planets:
        # Find the nearest planet we don't own
        nearest = None
        min_dist = float('inf')
        for t in targets:
            dist = math.sqrt((mine.x - t.x)**2 + (mine.y - t.y)**2)
            if dist < min_dist:
                min_dist = dist
                nearest = t

        if nearest is None:
            continue

        # How many ships do we need? Target's garrison + 1
        ships_needed = max(nearest.ships + 1, 20)

        # Only send if we have enough
        if mine.ships >= ships_needed:
            # Calculate angle from our planet to the target
            angle = math.atan2(nearest.y - mine.y, nearest.x - mine.x)
            moves.append([mine.id, angle, ships_needed])

    return moves

# **Trajectory Sniper** 

In [80]:
# Simulation-validated trajectory agent
# This replaces the old intercept() + safe_aim_angle() path.
def trajectory_sniper(obs, config=None):
    """
    Conservative greedy baseline:
    - Generate expansion and weak-frontier attack candidates.
    - Validate every shot by simulation.
    - Return nothing if no safe hitting trajectory exists.
    """
    state = get_state(obs)
    if not state.mine:
        return []

    candidates = []
    candidates += generate_expansion_candidates(state)
    candidates += generate_frontier_attack_candidates(state)

    if not candidates:
        return []

    return choose_non_conflicting_moves(candidates, max_moves=2)


# **Smart Sniper**

In [81]:
# Smart Snipe Policy Agent
# Priority hierarchy:
# 1. Snipe recently captured planets
# 2. Punish enemy overcommitment
# 3. Expand safely
# 4. Attack weak frontier planets
# 5. Wait

SNIPE_MEMORY = {}
SNIPE_MAX_TARGET_SHIPS = 45
SNIPE_MAX_TRAVEL_TIME = 30
SNIPE_EXTRA_OVERKILL = 4
PUNISH_SHIP_DROP = 25


def remember_previous_state(state):
    SNIPE_MEMORY[state.player] = {
        p.id: {
            "owner": p.owner,
            "ships": p.ships,
            "x": p.x,
            "y": p.y,
            "production": p.production,
        }
        for p in state.planets
    }


def previous_planet_snapshot(state, planet_id):
    return SNIPE_MEMORY.get(state.player, {}).get(planet_id)


def find_recent_enemy_captures(state):
    recent = []
    for p in state.enemy:
        old = previous_planet_snapshot(state, p.id)
        if old is None:
            continue
        if old["owner"] == -1 and p.owner != -1 and p.owner != state.player:
            recent.append(p)
    return recent


def find_enemy_overcommitments(state):
    weakened = []
    for p in state.enemy:
        old = previous_planet_snapshot(state, p.id)
        if old is None:
            continue
        same_enemy_owner = old["owner"] != -1 and old["owner"] != state.player and p.owner == old["owner"]
        large_drop = old["ships"] - p.ships >= PUNISH_SHIP_DROP
        if same_enemy_owner and large_drop:
            weakened.append(p)
    return weakened


def generate_snipe_candidates(state):
    candidates = []
    for target in find_recent_enemy_captures(state):
        if target.ships > SNIPE_MAX_TARGET_SHIPS:
            continue
        for source in candidate_sources(state.mine, target, state)[:3]:
            base = 14.0 * target.production + 5.0 * target.ships - 0.8 * distance(source, target)
            cand = validate_attack_candidate(
                state,
                source,
                target,
                "snipe_recent_capture",
                base_score=base,
                extra=SNIPE_EXTRA_OVERKILL,
            )
            if cand and cand["travel_time"] <= SNIPE_MAX_TRAVEL_TIME:
                candidates.append(cand)
    return candidates


def generate_punish_candidates(state):
    candidates = []
    for target in find_enemy_overcommitments(state):
        for source in candidate_sources(state.mine, target, state)[:3]:
            base = 10.0 * target.production + 4.0 * target.ships - 0.7 * distance(source, target)
            cand = validate_attack_candidate(
                state,
                source,
                target,
                "punish_overcommitment",
                base_score=base,
                extra=5,
            )
            if cand:
                candidates.append(cand)
    return candidates


def smart_snipe_agent(obs, config=None):
    """
    Main refined policy bot.
    Uses one shared validation rule: no shot is launched unless simulated path hits and avoids the sun.
    """
    state = get_state(obs)
    if not state.mine:
        remember_previous_state(state)
        return []

    # Priority 1: snipe planets the opponent just paid to capture.
    candidates = generate_snipe_candidates(state)
    if candidates:
        moves = choose_non_conflicting_moves(candidates, max_moves=1)
        remember_previous_state(state)
        return moves

    # Priority 2: punish a planet that just spent a large fleet.
    candidates = generate_punish_candidates(state)
    if candidates:
        moves = choose_non_conflicting_moves(candidates, max_moves=1)
        remember_previous_state(state)
        return moves

    # Priority 3/4: safe expansion, then weak frontier attacks.
    candidates = generate_expansion_candidates(state)
    if not candidates:
        candidates = generate_frontier_attack_candidates(state)

    moves = choose_non_conflicting_moves(candidates, max_moves=2) if candidates else []
    remember_previous_state(state)
    return moves


# Backward-compatible name from the previous prototype.
def delayed_snipe_agent(obs, config=None):
    return smart_snipe_agent(obs, config)


# **RL Class**

In [82]:
#RL Wrap
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from kaggle_environments import make


class OrbitWarsRLEnv(gym.Env):
    metadata = {"render_modes": ["human"], "render_fps": 4}

    def __init__(self, opponent="random", max_steps=400):
        super().__init__()
        self.opponent = opponent
        self.max_steps = max_steps

        self.action_space = spaces.Discrete(len(ACTION_MEANINGS))
        self.observation_space = spaces.Box(
            low=-1e6,
            high=1e6,
            shape=(OBS_SIZE,),
            dtype=np.float32,
        )

        self.env = None
        self.current_obs = None
        self.step_count = 0

        self.prev_my_planets = 0
        self.prev_enemy_planets = 0
        self.prev_my_ships = 0
        self.prev_enemy_ships = 0
        self.prev_my_prod = 0
        self.prev_enemy_prod = 0

    def _get_opponent_action(self, obs, config):
        if callable(self.opponent):
            return self.opponent(obs, config)
        return []

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.step_count = 0
        self.env = make("orbit_wars", debug=True)
        self.env.reset(num_agents=2)
        self.env.step([[], []])

        self.current_obs = self.env.state[0].observation

        _, _, my_planets, enemy_planets, neutral_planets, _ = parse_obs(self.current_obs)
        self.prev_my_planets = len(my_planets)
        self.prev_enemy_planets = len(enemy_planets)
        self.prev_my_ships = sum(p.ships for p in my_planets)
        self.prev_enemy_ships = sum(p.ships for p in enemy_planets)
        self.prev_my_prod = sum(p.production for p in my_planets)
        self.prev_enemy_prod = sum(p.production for p in enemy_planets)

        features = extract_features(self.current_obs)
        info = {}
        return features, info

    def step(self, action):
        self.step_count += 1

        my_obs = self.env.state[0].observation
        my_action, executed_action, used_fallback, failed_original, attempted_actions = choose_move_with_fallback(
            my_obs, int(action)
        )

        opp_obs = self.env.state[1].observation
        opp_action = self._get_opponent_action(opp_obs, self.env.configuration)

        self.env.step([my_action, opp_action])

        player_state = self.env.state[0]
        self.current_obs = player_state.observation
        features = extract_features(self.current_obs)

        reward = 0.0
        if failed_original:
            reward -= 0.01
        elif used_fallback:
            reward -= 0.003

        _, _, my_planets, enemy_planets, neutral_planets, _ = parse_obs(self.current_obs)

        my_planets_now = len(my_planets)
        enemy_planets_now = len(enemy_planets)
        my_ships_now = sum(p.ships for p in my_planets)
        enemy_ships_now = sum(p.ships for p in enemy_planets)
        my_prod_now = sum(p.production for p in my_planets)
        enemy_prod_now = sum(p.production for p in enemy_planets)

        reward += 0.02 * (my_planets_now - self.prev_my_planets)
        reward -= 0.02 * (enemy_planets_now - self.prev_enemy_planets)
        reward += 0.002 * (my_ships_now - self.prev_my_ships)
        reward -= 0.002 * (enemy_ships_now - self.prev_enemy_ships)
        reward += 0.01 * (my_prod_now - self.prev_my_prod)
        reward -= 0.01 * (enemy_prod_now - self.prev_enemy_prod)

        self.prev_my_planets = my_planets_now
        self.prev_enemy_planets = enemy_planets_now
        self.prev_my_ships = my_ships_now
        self.prev_enemy_ships = enemy_ships_now
        self.prev_my_prod = my_prod_now
        self.prev_enemy_prod = enemy_prod_now

        terminated = False
        truncated = False

        if player_state.status != "ACTIVE":
            terminated = True
            if player_state.reward is not None:
                reward += float(player_state.reward)

        if self.step_count >= self.max_steps:
            truncated = True

        info = {
            "status": player_state.status,
            "raw_reward": player_state.reward,
            "my_planets": my_planets_now,
            "enemy_planets": enemy_planets_now,
            "my_ships": my_ships_now,
            "enemy_ships": enemy_ships_now,
            "my_prod": my_prod_now,
            "enemy_prod": enemy_prod_now,
            "chosen_action": int(action),
            "executed_action": int(executed_action),
            "used_fallback": used_fallback,
            "failed_original": failed_original,
            "attempted_actions": attempted_actions,
        }

        return features, reward, terminated, truncated, info

    def render(self):
        if self.env is not None:
            return self.env.render(mode="ipython", width=800, height=600)

In [83]:
#Training
from stable_baselines3 import PPO

train_env = OrbitWarsRLEnv(opponent=random_like_macro_agent, max_steps=100)

model = PPO(
    "MlpPolicy",
    train_env,
    verbose=1,
    n_steps=256,
    batch_size=64,
    learning_rate=3e-4,
    gamma=0.99,
)

model.learn(total_timesteps=5000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 100      |
|    ep_rew_mean     | -1.82    |
| time/              |          |
|    fps             | 99       |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 256      |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 100         |
|    ep_rew_mean          | -1.68       |
| time/                   |             |
|    fps                  | 105         |
|    iterations           | 2           |
|    time_elapsed         | 4           |
|    total_timesteps      | 512         |
| train/                  |             |
|    approx_kl            | 0.011259321 |
|    clip_fraction        | 0.0582      |
|    clip_range           | 0.2         |
|    entropy_loss   

# **Testing Area**

# Making a submission

You can either submit a main.py, a tar.gz (or zip) with a main.py in it, or submit a notebook with a main.py or submission.tar.gz

There are three ways to subit.
1. using the [Submit Agent](https://www.kaggle.com/competitions/orbit-wars) button on the homepage and uploading the file
2. using the Kaggle CLI (as described in agents.py in the competition dataset)
3. submitting a notebook with a main.py or submission.tar.gz

In [89]:
#Submission
%%writefile main.py
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet

def nearest_planet_sniper(obs):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]

    # Separate our planets from targets
    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]

    if not targets:
        return moves

    for mine in my_planets:
        # Find the nearest planet we don't own
        nearest = None
        min_dist = float('inf')
        for t in targets:
            dist = math.sqrt((mine.x - t.x)**2 + (mine.y - t.y)**2)
            if dist < min_dist:
                min_dist = dist
                nearest = t

        if nearest is None:
            continue

        # How many ships do we need? Target's garrison + 1
        ships_needed = max(nearest.ships + 1, 20)

        # Only send if we have enough
        if mine.ships >= ships_needed:
            # Calculate angle from our planet to the target
            angle = math.atan2(nearest.y - mine.y, nearest.x - mine.x)
            moves.append([mine.id, angle, ships_needed])

    return moves

UsageError: Line magic function `%%writefile` not found.


# **Debugger**

In [ ]:
import matplotlib.pyplot as plt


def debug_candidate_shot(obs, source_id=None, target_id=None, ships=30):
    state = get_state(obs)

    if source_id is None:
        source = max(state.mine, key=lambda p: p.ships)
    else:
        source = next(p for p in state.mine if p.id == source_id)

    possible_targets = state.neutral + state.enemy

    if target_id is None:
        target = min(
            possible_targets,
            key=lambda p: math.hypot(p.x - source.x, p.y - source.y),
        )
    else:
        target = next(p for p in state.planets if p.id == target_id)

    angle, result = find_safe_hitting_angle(
        source,
        target,
        ships,
        state.angular_velocity,
    )

    print("source:", source.id, "target:", target.id)
    print("angle:", angle)
    print("result:", result)

    fig, ax = plt.subplots(figsize=(7, 7))

    # Sun
    sun = plt.Circle((SUN_X, SUN_Y), SUN_RADIUS + SUN_BUFFER, fill=False)
    ax.add_patch(sun)
    ax.scatter([SUN_X], [SUN_Y], s=100, label="Sun")

    # Source and current target
    ax.scatter([source.x], [source.y], s=80, label=f"Source {source.id}")
    ax.scatter([target.x], [target.y], s=80, label=f"Target now {target.id}")

    # Target future path
    txs = []
    tys = []
    for t in np.arange(0, MAX_SIM_TURNS, DT):
        tx, ty = predict_planet_position(target, t, state.angular_velocity)
        txs.append(tx)
        tys.append(ty)
    ax.plot(txs, tys, linestyle="--", label="Predicted target path")

    # Fleet path
    if angle is not None:
        speed = fleet_speed(ships)
        fxs = []
        fys = []
        for t in np.arange(0, min(MAX_SIM_TURNS, result["best_t"] + 5), DT):
            fxs.append(source.x + speed * t * math.cos(angle))
            fys.append(source.y + speed * t * math.sin(angle))
        ax.plot(fxs, fys, label="Fleet path")

        ax.scatter(
            [result["fleet_x"]],
            [result["fleet_y"]],
            s=100,
            marker="x",
            label="Closest fleet point",
        )
        ax.scatter(
            [result["target_x"]],
            [result["target_y"]],
            s=100,
            marker="x",
            label="Closest target point",
        )

    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True)
    ax.legend()
    plt.show()

In [ ]:
# Quick sanity tests
# 1. Visualize one simulated candidate shot.
env = make("orbit_wars", debug=True)
env.reset(num_agents=2)
env.step([[], []])
obs = env.state[0].observation
debug_candidate_shot(obs, ships=30)

# 2. Run the refined policy bot.
env = make("orbit_wars", debug=True)
env.run([smart_snipe_agent, "random"])
final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")
env.render(mode="ipython", width=800, height=600)
